In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers
!pip install torch==2.2.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 150.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 122.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 114.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 66.2 MB/s eta 0:00:

In [1]:
!pip install transformers==4.41.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 147.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2


In [2]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 138.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1

In [3]:
# =====================================================
#   SENTIMENT ANALYSIS + CATEGORY ASSIGNMENT (REDDIT)
#   Batched GPU version
# =====================================================

from transformers import pipeline, AutoTokenizer
import pandas as pd
from datetime import datetime, timezone
from google.colab import files
import torch

print("=== Starting Reddit Sentiment Analysis (Batched GPU) ===\n", flush=True)
print(f"GPU available: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}\n", flush=True)


# ─── GoEmotions Analyzer (batched) ─────────────────
class GoEmotionsAnalyzer:
    def __init__(self, batch_size=64):
        print("Loading GoEmotions model...", flush=True)
        self.batch_size = batch_size
        self.device = 0 if torch.cuda.is_available() else -1
        self.model = pipeline(
            "text-classification",
            model="SamLowe/roberta-base-go_emotions",
            return_all_scores=True,
            device=self.device,
            batch_size=self.batch_size
        )
        self.tokenizer = AutoTokenizer.from_pretrained("SamLowe/roberta-base-go_emotions")
        device_label = f"GPU ({torch.cuda.get_device_name(0)})" if self.device == 0 else "CPU"
        print(f"Model loaded on {device_label}, batch_size={batch_size}\n", flush=True)

    def _parse_scores(self, raw):
        emotion_dict = {item["label"]: item["score"] for item in raw if "label" in item and "score" in item}

        if emotion_dict:
            top_emotion = max(emotion_dict, key=emotion_dict.get)
            top_score = emotion_dict[top_emotion]
            emotion_dict = {top_emotion: top_score} if top_score >= 0.5 else {}
        else:
            emotion_dict = {}

        top_emotion = max(emotion_dict, key=emotion_dict.get, default="neutral")
        label = (
            "positive" if top_emotion in ["joy","admiration","amusement","approval","caring","desire","excitement","gratitude","love","optimism","pride","relief"]
            else "negative" if top_emotion in ["anger","annoyance","disappointment","disapproval","disgust","fear","grief","remorse","sadness"]
            else "neutral"
        )
        return {"label": label, "top_emotion": top_emotion, "score": emotion_dict.get(top_emotion, 0.0)}, emotion_dict

    def analyze_batch(self, texts):
        safe = [t if (t and isinstance(t, str) and t.strip()) else "neutral" for t in texts]
        raw_results = self.model(safe, truncation=True, max_length=512)

        parsed = []
        for i, raw in enumerate(raw_results):
            if not texts[i] or not str(texts[i]).strip():
                parsed.append(({"label": "neutral", "top_emotion": "neutral", "score": 0.0}, {}))
            else:
                parsed.append(self._parse_scores(raw))
        return parsed


# ─── 10 Emotion Buckets ────────────────────────────
def bucket_emotions(top_emotion):
    mapping = {
        "joy": "joy_amusement", "amusement": "joy_amusement", "optimism": "joy_amusement",
        "love": "love_admiration", "admiration": "love_admiration", "approval": "love_admiration",
        "gratitude": "gratitude_caring", "caring": "gratitude_caring",
        "excitement": "passion_hype",
        "anger": "anger_frustration", "annoyance": "anger_frustration",
        "disapproval": "anger_frustration", "disgust": "anger_frustration",
        "sadness": "sadness_disappointment", "disappointment": "sadness_disappointment",
        "grief": "sadness_disappointment", "remorse": "sadness_disappointment",
        "fear": "fear_nervousness", "nervousness": "fear_nervousness",
        "surprise": "surprise_confusion", "confusion": "surprise_confusion",
        "curiosity": "surprise_confusion", "realization": "surprise_confusion",
        "neutral": "neutral"
    }
    return mapping.get(top_emotion, "other")


# ─── Subreddit → Category mapping ──────────────────
SUBREDDIT_TO_CATEGORY = {
    "gaming": "Gaming", "games": "Gaming", "pcgaming": "Gaming",
    "leagueoflegends": "Gaming", "globaloffensive": "Gaming",
    "funny": "Entertainment", "entertainment": "Entertainment",
    "movies": "Entertainment", "television": "Entertainment",
    "music": "Music", "hiphopheads": "Music", "indieheads": "Music",
    "memes": "Comedy", "dankmemes": "Comedy", "jokes": "Comedy",
    "technology": "Science & Technology", "science": "Science & Technology",
    "machinelearning": "Science & Technology", "artificial": "Science & Technology",
    "askreddit": "AskReddit",
}

def subreddit_to_category(subreddit: str) -> str:
    return SUBREDDIT_TO_CATEGORY.get(subreddit.lower(), subreddit)


# ─── Load & Clean CSV ───────────────────────────────
CSV_PATH = "/content/top5_comments.csv"
df_raw = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df_raw):,} rows", flush=True)

df_raw = df_raw.dropna(subset=["comment_body"])
df_raw = df_raw[df_raw["comment_body"].str.strip() != ""].reset_index(drop=True)
print(f"After dropping empty comments: {len(df_raw):,} rows\n", flush=True)


# ─── Batched Analysis Loop ──────────────────────────
BATCH_SIZE = 64
analyzer = GoEmotionsAnalyzer(batch_size=BATCH_SIZE)

texts = df_raw["comment_body"].tolist()
all_sentiments = []
total = len(texts)

print(f"Processing {total:,} comments in batches of {BATCH_SIZE}...\n", flush=True)
start_time = datetime.now()

for i in range(0, total, BATCH_SIZE):
    batch_texts = texts[i : i + BATCH_SIZE]
    results = analyzer.analyze_batch(batch_texts)
    all_sentiments.extend(results)

    elapsed = (datetime.now() - start_time).seconds
    pct = min((i + BATCH_SIZE) / total * 100, 100)
    rate = (i + BATCH_SIZE) / max(elapsed, 1)
    remaining = int((total - i - BATCH_SIZE) / max(rate, 1))
    print(f"  {pct:.1f}% — {min(i + BATCH_SIZE, total):,}/{total:,} | {rate:.0f} rows/sec | ~{remaining}s remaining", flush=True)

elapsed_total = (datetime.now() - start_time).seconds
print(f"\n✅ Inference complete in {elapsed_total}s ({total / max(elapsed_total, 1):.0f} rows/sec avg)\n", flush=True)


# ─── Build Output DataFrame ─────────────────────────
print("Building output dataframe...", flush=True)
all_results = []
for idx, (sent, emo) in enumerate(all_sentiments):
    row = df_raw.iloc[idx]
    text = str(row["comment_body"])
    all_results.append({
        "post_id":           row["post_id"],
        "subreddit":         row["subreddit"],
        "category":          subreddit_to_category(row["subreddit"]),
        "post_title":        row["post_title"],
        "comment_id":        row["comment_id"],
        "comment_author":    row.get("comment_author", ""),
        "comment_score":     row.get("comment_score", None),
        "source":            "comment",
        "text":              text[:180] + "..." if len(text) > 180 else text,
        "sentiment_label":   sent["label"],
        "top_emotion":       sent["top_emotion"],
        "top_emotion_score": sent["score"],
        "emotion_bucket":    bucket_emotions(sent["top_emotion"]),
        "processedAt":       datetime.now(timezone.utc).isoformat() + "Z"
    })

df_out = pd.DataFrame(all_results)
out_file = "reddit_sentiment_with_category.csv"
df_out.to_csv(out_file, index=False)

print(f"File: {out_file} ({len(df_out):,} rows)", flush=True)
display(df_out.head(10))
files.download(out_file)

print("\nCategory Summary:", flush=True)
display(df_out.groupby("category").size().reset_index(name="count"))
print("\nSentiment Summary:", flush=True)
display(df_out.groupby("sentiment_label").size().reset_index(name="count"))

=== Starting Reddit Sentiment Analysis (Batched GPU) ===

GPU available: True
Device: NVIDIA L4

Loaded 113,920 rows
After dropping empty comments: 113,907 rows

Loading GoEmotions model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

<IPython.core.display.Javascript object>


Category Summary:


,category,count
0,AskReddit,89157
1,Comedy,5092
2,Entertainment,5584
3,Gaming,2148
4,Music,2003
5,Science & Technology,558
6,aww,2854
7,todayilearned,2012
8,worldnews,4499



Sentiment Summary:


,sentiment_label,count
0,negative,8808
1,neutral,87050
2,positive,18049
